# 02 — Preprocessing Pipeline

Frame extraction, SSIM mask, landmark detection, cutout operations, augmentation.

**Requires:** FF++ data, `shape_predictor_68_face_landmarks.dat`.

In [ ]:
import os, random
import numpy as np
import cv2
import dlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from skimage.metrics import structural_similarity as ssim
from albumentations import (
    Compose, GaussNoise, GaussianBlur, HorizontalFlip,
    OneOf, RandomBrightnessContrast, FancyPCA,
    HueSaturationValue, ShiftScaleRotate
)

## Constants

In [ ]:
IMG_SIZE = 224
FRAME_STEP = 3
FRAME_COUNT = 12

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# SSIM polygon selection
SSIM_BIN = 0.5
MIN_OVERLAP = 0.0
DEFAULT_POLY_MODE = "sim"   # always cut the most similar region
MIN_AREA_RATIO = 0.02
MAX_AREA_RATIO = 0.05

# optional polygon scaling
POLY_SCALE_TO_TARGET = False
TARGET_AREA_RATIO = 0.08

# star cutout on real frames
STAR_MIN = 8
STAR_MAX = 16

PREDICTOR_PATH = "/content/drive/MyDrive/FF_Data/shape_predictor_68_face_landmarks.dat"

## Frame Extraction

In [ ]:
def extract_frames(path_a, path_b, frame_step=FRAME_STEP,
                   frame_count=FRAME_COUNT, img_size=IMG_SIZE):
    cap_a, cap_b = cv2.VideoCapture(path_a), cv2.VideoCapture(path_b)

    total = min(int(cap_a.get(cv2.CAP_PROP_FRAME_COUNT)),
                int(cap_b.get(cv2.CAP_PROP_FRAME_COUNT)))

    if total >= frame_count * frame_step:
        indices = [i * frame_step for i in range(frame_count)]
    else:
        indices = np.linspace(0, max(0, total - 1), frame_count, dtype=int).tolist()

    frames_a, frames_b = [], []
    for idx in indices:
        cap_a.set(cv2.CAP_PROP_POS_FRAMES, idx)
        cap_b.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok_a, fa = cap_a.read()
        ok_b, fb = cap_b.read()

        if ok_a and ok_b:
            fa = cv2.cvtColor(cv2.resize(fa, (img_size, img_size)), cv2.COLOR_BGR2RGB)
            fb = cv2.cvtColor(cv2.resize(fb, (img_size, img_size)), cv2.COLOR_BGR2RGB)
            frames_a.append(fa); frames_b.append(fb)
        elif frames_a:
            frames_a.append(frames_a[-1].copy())
            frames_b.append(frames_b[-1].copy())
        else:
            black = np.zeros((img_size, img_size, 3), dtype=np.uint8)
            frames_a.append(black.copy()); frames_b.append(black.copy())

    cap_a.release(); cap_b.release()

    while len(frames_a) < frame_count:
        frames_a.append(frames_a[-1].copy())
        frames_b.append(frames_b[-1].copy())

    return frames_a, frames_b

## SSIM Difference Mask

In [ ]:
def compute_ssim_mask(real_frame, fake_frame):
    r = real_frame.astype(np.float32) / 255.0
    f = fake_frame.astype(np.float32) / 255.0
    diff = np.zeros((r.shape[0], r.shape[1]), dtype=np.float32)
    for c in range(3):
        _, smap = ssim(r[:, :, c], f[:, :, c], full=True, data_range=1.0)
        diff += (1.0 - smap)
    diff /= 3.0
    return np.clip(diff, 0, 1)

## Landmark Detection

In [ ]:
face_detector = dlib.get_frontal_face_detector()
shape_predictor = dlib.shape_predictor(PREDICTOR_PATH)

def extract_landmarks(frame, verbose=False):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    faces = face_detector(gray)
    if not faces:
        if verbose: print("No face detected.")
        return None
    shape = shape_predictor(gray, faces[0])
    return [(p.x, p.y) for p in shape.parts()]

REGION_IDX = {
    "jaw":           list(range(0, 17)),
    "right_eyebrow": list(range(17, 22)),
    "left_eyebrow":  list(range(22, 27)),
    "nose":          list(range(27, 36)),
    "right_eye":     list(range(36, 42)),
    "left_eye":      list(range(42, 48)),
    "outer_mouth":   list(range(48, 60)),
    "inner_mouth":   list(range(60, 68)),
}

## Polygon Selection

Generate candidate polygons from landmarks. Select the one with highest overlap to the SSIM similarity region. Includes fallback if no polygon passes the overlap threshold.

In [ ]:
def polygon_area(poly):
    return cv2.contourArea(np.array(poly, dtype=np.float32))

def scale_polygon(poly, scale):
    cnt = np.array(poly, dtype=np.float32)
    c = cnt.mean(axis=0, keepdims=True)
    scaled = (cnt - c) * float(scale) + c
    return scaled.astype(np.int32).tolist()

def generate_candidate_polygons(landmarks, n_random=6):
    if landmarks is None or len(landmarks) != 68:
        return []

    polys = [
        landmarks[36:42], landmarks[42:48],   # eyes
        landmarks[27:36],                      # nose
        landmarks[48:60], landmarks[60:68],    # mouth
    ]

    all_idx = list(range(68))
    for _ in range(n_random):
        k = random.randint(5, 12)
        sel = random.sample(all_idx, k)
        pts = [landmarks[i] for i in sel]
        hull = cv2.convexHull(np.array(pts, dtype=np.int32)).squeeze().tolist()
        if isinstance(hull[0], int):
            hull = [hull]
        polys.append(hull)

    return polys


def select_best_polygon(polygons, ssim_mask, min_overlap=MIN_OVERLAP,
                        ssim_bin=SSIM_BIN, mode=DEFAULT_POLY_MODE,
                        verbose=False):
    H, W = ssim_mask.shape[:2]
    img_area = float(H * W)

    # build interest region mask
    if mode == "diff":
        region = (ssim_mask >= ssim_bin).astype(np.uint8)
    elif mode == "sim":
        region = (ssim_mask <= (1.0 - ssim_bin)).astype(np.uint8)
    else:
        raise ValueError("mode must be 'diff' or 'sim'")

    mask_total = int(region.sum())

    best_polygon, best_rho, best_overlap = None, -1.0, -1

    # fallback candidate (best regardless of threshold)
    fb_polygon, fb_rho, fb_overlap, fb_area = None, -1.0, -1, -1.0

    for idx, poly in enumerate(polygons):
        try:
            area = polygon_area(poly)
        except Exception:
            continue

        area_ratio = area / img_area if img_area > 0 else 0.0
        if area_ratio < MIN_AREA_RATIO or area_ratio > MAX_AREA_RATIO:
            continue

        pmask = np.zeros_like(region, dtype=np.uint8)
        try:
            cv2.fillPoly(pmask, [np.array(poly, dtype=np.int32)], 1)
        except Exception:
            continue

        overlap_area = int(((pmask == 1) & (region == 1)).sum())
        rho = overlap_area / mask_total if mask_total > 0 else 0.0

        # primary selection: passes threshold
        if mask_total > 0 and rho >= min_overlap:
            if rho > best_rho or (np.isclose(rho, best_rho) and overlap_area > best_overlap):
                best_polygon, best_rho, best_overlap = poly, rho, overlap_area

        # fallback tracking
        if (rho, overlap_area, area) > (fb_rho, fb_overlap, fb_area):
            fb_polygon, fb_rho, fb_overlap, fb_area = poly, rho, overlap_area, area

    # fallback if nothing passed threshold
    if best_polygon is None and fb_polygon is not None:
        best_polygon = fb_polygon

    # optional: scale to target area
    if best_polygon is not None and POLY_SCALE_TO_TARGET and TARGET_AREA_RATIO and img_area > 0:
        cur_ratio = polygon_area(best_polygon) / img_area
        if cur_ratio > 0:
            s = (TARGET_AREA_RATIO / cur_ratio) ** 0.5
            best_polygon = scale_polygon(best_polygon, s)

    return best_polygon

## Cutout Operations

Three fill strategies for polygon cutout on fake frames. Star cutout on real frames prevents overfitting.

In [ ]:
def make_fill(h, w, fill_type):
    if fill_type == "zero":
        return np.zeros((h, w, 3), dtype=np.uint8)
    if fill_type == "white":
        return np.full((h, w, 3), 255, dtype=np.uint8)
    return np.random.randint(0, 256, (h, w, 3), dtype=np.uint8)

def apply_polygon_cutout(frame, polygon, fill_type="zero"):
    if polygon is None:
        return frame.copy()
    h, w = frame.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [np.array(polygon, dtype=np.int32)], 1)
    fill = make_fill(h, w, fill_type)
    out = frame.copy()
    out[mask == 1] = fill[mask == 1]
    return out

def apply_seferbekov_cutout(real_frames, fake_frames, polygon,
                            fake_fill_type="zero"):
    cut_real = [f.copy() for f in real_frames]
    if polygon is None:
        return cut_real, [f.copy() for f in fake_frames]
    cut_fake = []
    for f in fake_frames:
        fr = f if f.dtype == np.uint8 else np.clip(f * 255, 0, 255).astype(np.uint8)
        cut_fake.append(apply_polygon_cutout(fr, polygon, fill_type=fake_fill_type))
    return cut_real, cut_fake

def random_star_polygon(h, w, min_r=STAR_MIN, max_r=STAR_MAX, points=5):
    R = np.random.randint(min_r, max_r)
    r = max(3, int(R * 0.5))
    cx = np.random.randint(R, w - R)
    cy = np.random.randint(R, h - R)
    pts = []
    for i in range(points * 2):
        ang = np.pi * i / points
        rad = R if (i % 2 == 0) else r
        pts.append((int(cx + rad * np.cos(ang)), int(cy + rad * np.sin(ang))))
    return pts

def apply_star_on_real_frames(real_frames, enable=True, prob=0.5,
                              star_fill_type="zero"):
    if not enable:
        return [f.copy() for f in real_frames]
    out = []
    for f in real_frames:
        if np.random.rand() < prob:
            h, w = f.shape[:2]
            star = random_star_polygon(h, w)
            out.append(apply_polygon_cutout(f, star, fill_type=star_fill_type))
        else:
            out.append(f.copy())
    return out

## Augmentation and Normalization

In [ ]:
def create_augmentations(high_prob=False):
    p = 0.2 if high_prob else 0.05
    p_geo = 0.5 if high_prob else 0.1
    p_color = 0.5 if high_prob else 0.1

    return Compose([
        GaussNoise(p=p),
        GaussianBlur(blur_limit=(3, 7), p=p),
        HorizontalFlip(),
        OneOf([
            RandomBrightnessContrast(brightness_limit=0.05, contrast_limit=0.05),
            FancyPCA(alpha=0.05),
            HueSaturationValue(hue_shift_limit=5, sat_shift_limit=5, val_shift_limit=5)
        ], p=p_color),
        ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=5,
                         border_mode=cv2.BORDER_CONSTANT, p=p_geo),
    ], additional_targets={"image2": "image"})

def resize_and_normalize(frame1, frame2, augment=True, augment_instance=None):
    """Augment on uint8, then ImageNet-normalize."""
    f1_u8 = frame1 if frame1.dtype == np.uint8 else np.clip(frame1 * 255, 0, 255).astype(np.uint8)
    f2_u8 = frame2 if frame2.dtype == np.uint8 else np.clip(frame2 * 255, 0, 255).astype(np.uint8)

    if augment and augment_instance is not None:
        out = augment_instance(image=f1_u8, image2=f2_u8)
        f1_u8, f2_u8 = out["image"], out["image2"]

    f1 = f1_u8.astype(np.float32) / 255.0
    f2 = f2_u8.astype(np.float32) / 255.0
    for i in range(3):
        f1[:, :, i] = (f1[:, :, i] - MEAN[i]) / STD[i]
        f2[:, :, i] = (f2[:, :, i] - MEAN[i]) / STD[i]

    return f1.astype(np.float32), f2.astype(np.float32)

def denormalize(img):
    x = img.astype(np.float32).copy()
    for i in range(3):
        x[..., i] = (x[..., i] * STD[i]) + MEAN[i]
    return np.clip(x * 255, 0, 255).astype(np.uint8)

## Full Pipeline Visualization

In [ ]:
# Run after notebook 01 sets video_pairs, train_pairs etc.
real_path, fake_path = video_pairs[0]
real_frames, fake_frames = extract_frames(real_path, fake_path)
mid = len(real_frames) // 2

# SSIM mask
mask = compute_ssim_mask(real_frames[mid], fake_frames[mid])

# Polygon selection
lm = extract_landmarks(real_frames[mid])
polys = generate_candidate_polygons(lm)
best = select_best_polygon(polys, mask, mode="sim", verbose=True)

# Cutout on fake, star on real
fake_cut = apply_polygon_cutout(fake_frames[mid], best, fill_type="zero")
real_star = apply_star_on_real_frames([real_frames[mid]], prob=1.0, star_fill_type="zero")[0]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0,0].imshow(real_frames[mid]);  axes[0,0].set_title("Real (raw)")
axes[0,1].imshow(fake_frames[mid]);  axes[0,1].set_title("Fake (raw)")
axes[0,2].imshow(mask, cmap="hot");  axes[0,2].set_title("SSIM diff mask")
axes[1,0].imshow(real_star);         axes[1,0].set_title("Real + star cutout")
axes[1,1].imshow(fake_cut);          axes[1,1].set_title("Fake + polygon cutout")
axes[1,2].imshow(denormalize(resize_and_normalize(fake_cut, fake_cut, augment=False)[0]))
axes[1,2].set_title("Normalized (display)")
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()